# 🔧 Kube SRE Gym — GRPO Training on Colab

Train a Kubernetes SRE agent using GRPO (Group Relative Policy Optimization) with TRL.

The agent learns to diagnose and fix real K8s incidents:
- OOMKill, CrashLoopBackOff, ImagePullBackOff
- Bad config, liveness probe failures, resource quotas
- Cascading multi-service failures

**Architecture:**
- Environment server runs on HF Spaces (talks to real GKE cluster)
- Claude judges agent actions via Anthropic API
- Agent model (Qwen3-8B) trains here via GRPO with LoRA

**Requirements:** Colab T4/A100 GPU, HF Spaces env server running

## 1. Install Dependencies

In [ ]:
!pip install -q \
    "openenv-core[core] @ git+https://github.com/meta-pytorch/OpenEnv.git@v0.2.1" \
    "trl[vllm]==0.29.0" \
    "vllm==0.11.2" \
    "peft" \
    "transformers" \
    "datasets" \
    "huggingface_hub>=0.20.0" \
    "tabulate>=0.9.0" \
    "anthropic>=0.39.0" \
    "torch>=2.8.0"

## 2. Install Kube SRE Gym Client

In [ ]:
!pip install -q git+https://huggingface.co/spaces/openenv-community/kube-sre-gym

## 3. Configuration

Set your environment server URL and HuggingFace token.

In [ ]:
import os
from google.colab import userdata

# Environment server URL (running on HF Spaces or your H100)
ENV_URL = "https://openenv-community-kube-sre-gym.hf.space"  # HF Spaces
# ENV_URL = "http://localhost:8000"  # Local server

# HuggingFace token for pushing model checkpoints
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Model configuration
MODEL_ID = "Qwen/Qwen3-8B"
HUB_REPO = "openenv-community/k8s-sre-agent-qwen3-8b"  # Where to push trained model
NUM_EPISODES = 50
NUM_GENERATIONS = 4  # GRPO rollouts per episode
MAX_TURNS = 15  # Max commands per episode

print(f"Environment: {ENV_URL}")
print(f"Model: {MODEL_ID}")
print(f"Episodes: {NUM_EPISODES}, Generations: {NUM_GENERATIONS}")

## 4. Test Environment Connection

Verify the environment server is reachable and working.

In [ ]:
from kube_sre_gym import KubeSreGymEnv, KubeSreGymAction

# Quick connectivity test
with KubeSreGymEnv(base_url=ENV_URL) as env:
    result = env.reset()
    print("✅ Connected to environment!")
    print(f"Alert: {result.observation.command_output[:200]}")
    print(f"\nCluster status:\n{result.observation.cluster_status_summary[:500]}")
    
    # Test a command
    result = env.step(KubeSreGymAction(command="kubectl get pods -A"))
    print(f"\nStep result (reward={result.reward}):\n{result.observation.command_output[:500]}")

## 5. System Prompt & Helpers

The system prompt gives the agent K8s SRE knowledge: cluster topology, available commands, common fixes.

In [ ]:
import csv
import json
import logging
from datetime import datetime
from pathlib import Path

from datasets import Dataset
from transformers import AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer
from trl.experimental.openenv import generate_rollout_completions

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

SYSTEM_PROMPT = """You are an expert Kubernetes SRE (Site Reliability Engineer).
You receive PagerDuty alerts about Kubernetes incidents and must diagnose and fix them.

IMPORTANT RULES:
- Always specify namespace with -n <namespace> (pods are in: payments, frontend, auth)
- Start with: kubectl get pods -n <namespace> to see actual pod names and status
- Never guess pod names — always list pods first, then use exact names from the output
- NEVER repeat a command you already ran — check your previous commands first
- Output exactly ONE command. No explanations, no extra text. Just the command.

CLUSTER TOPOLOGY:
- payments namespace: payment-gateway (container: payment-gateway, image: nginx:1.25), payment-worker (container: payment-worker, image: busybox:1.36), payment-api (container: payment-api, image: python:3.11-slim)
- frontend namespace: web-app (container: web-app, image: nginx:1.25), frontend-cache (container: frontend-cache, image: redis:7)
- auth namespace: auth-service (container: auth-service, image: nginx:1.25)

AVAILABLE COMMANDS:
- kubectl get pods/deployments/events/services -n <ns>
- kubectl describe pod/deployment <name> -n <ns>
- kubectl logs <pod-name> -n <ns> [--previous]
- kubectl set image deployment/<name> <container>=<image> -n <ns>
- kubectl set resources deployment/<name> -c <container> --limits=memory=<val> -n <ns>
- kubectl set env deployment/<name> KEY=VALUE -n <ns>
- kubectl patch deployment <name> -n <ns> -p '{"spec":...}'
- kubectl rollout restart deployment/<name> -n <ns>
- kubectl scale deployment/<name> --replicas=N -n <ns>
- kubectl delete pod <name> -n <ns>

COMMON FIXES (use the exact container names from CLUSTER TOPOLOGY above):
- CrashLoopBackOff (bad command): kubectl patch deployment <name> -n <ns> -p '{"spec":{"template":{"spec":{"containers":[{"name":"<container>","command":null,"args":null}]}}}}'
- OOMKilled (exit code 137): kubectl set resources deployment/<name> -c <container> --limits=memory=256Mi -n <ns>
- ImagePullBackOff: kubectl set image deployment/<name> <container>=<correct-image-from-topology> -n <ns>
- Bad env config: kubectl set env deployment/<name> KEY=CORRECT_VALUE -n <ns>
- Liveness probe wrong path: kubectl patch deployment <name> -n <ns> -p '{"spec":{"template":{"spec":{"containers":[{"name":"<container>","livenessProbe":{"httpGet":{"path":"/","port":80}}}]}}}}'

WORKFLOW: list pods → describe/logs the broken pod → diagnose: <root cause> → fix: kubectl <command>
After applying a fix, STOP. Do not repeat the fix or run more commands."""


def format_observation(obs) -> str:
    """Format observation into agent-readable text."""
    command_output = getattr(obs, "command_output", "") or ""
    cluster_status = getattr(obs, "cluster_status_summary", "") or ""
    hint = getattr(obs, "hint", "") or ""
    steps = getattr(obs, "steps_taken", 0)
    max_steps = getattr(obs, "max_steps", 15)
    text = f"{command_output}\n\nCURRENT CLUSTER STATUS:\n{cluster_status}"
    if hint:
        text += f"\n\nHINT: {hint}"
    text += f"\n\nStep {steps}/{max_steps}. Diagnose and fix this incident."
    return text


def format_history(history: list[dict]) -> str:
    """Format conversation history for the agent."""
    if not history:
        return ""
    lines = ["PREVIOUS COMMANDS AND RESULTS:"]
    for entry in history:
        output = entry["output"]
        if len(output) > 300:
            output = output[:300] + "... (truncated)"
        lines.append(f"$ {entry['command']}")
        lines.append(f"  Output: {output}")
        if entry.get("feedback"):
            lines.append(f"  Feedback: {entry['feedback']}")
    return "\n".join(lines)


def parse_commands(text: str) -> list[str]:
    """Extract kubectl/diagnose/fix commands from agent response."""
    commands = []
    seen = set()
    for line in text.strip().split("\n"):
        line = line.strip()
        if line.startswith(("kubectl ", "diagnose:", "fix:")):
            if line not in seen:
                commands.append(line)
                seen.add(line)
        elif line.startswith(("- kubectl", "* kubectl", "> kubectl")):
            cmd = line.lstrip("-*> ")
            if cmd not in seen:
                commands.append(cmd)
                seen.add(cmd)
        if len(commands) >= 2:
            break
    return commands


def apply_chat_template(tokenizer, messages):
    """Apply chat template with fallback."""
    try:
        return tokenizer.apply_chat_template(
            messages, add_generation_prompt=True,
            tokenize=False, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False)


print("✅ Helpers loaded")

## 6. Rollout Function

Each rollout runs one full SRE episode: the agent receives an alert, investigates, diagnoses, and fixes the incident on a real GKE cluster.

In [ ]:
def rollout_once(trainer, env, tokenizer, system_prompt, max_turns):
    """Run one full K8s incident episode."""
    result = env.reset()
    observation = result.observation

    prompt_ids, completion_ids, logprobs = [], [], []
    step_rewards, diagnosis_rewards, fix_rewards = [], [], []
    conversation_history = []

    for _turn in range(max_turns):
        if result.done:
            break

        history_text = format_history(conversation_history)
        obs_text = format_observation(observation)
        user_prompt = f"{history_text}\n\n---\n\nCURRENT OBSERVATION:\n{obs_text}" if history_text else obs_text

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        prompt_text = apply_chat_template(tokenizer, messages)

        # Generate with vLLM via TRL
        rollout_outputs = generate_rollout_completions(trainer, [prompt_text])[0]
        prompt_ids.extend(rollout_outputs["prompt_ids"])
        completion_ids.extend(rollout_outputs["completion_ids"])
        logprobs.extend(rollout_outputs["logprobs"])

        completion_text = rollout_outputs.get("text") or tokenizer.decode(
            rollout_outputs["completion_ids"], skip_special_tokens=True)

        commands = parse_commands(completion_text)
        if not commands:
            step_rewards.append(-0.5)
            conversation_history.append({
                "command": completion_text[:100].strip(),
                "output": "(no valid command parsed)",
                "reward": -0.5,
                "feedback": "Invalid output — expected kubectl/diagnose:/fix: command.",
            })
            continue

        for cmd in commands:
            try:
                result = env.step(KubeSreGymAction(command=cmd))
                reward = float(result.reward or 0.0)
                step_rewards.append(reward)
                observation = result.observation
                cmd_output = getattr(observation, "command_output", "") or ""
                hint = getattr(observation, "hint", "") or ""
                conversation_history.append({
                    "command": cmd, "output": cmd_output[:500],
                    "reward": reward, "feedback": hint,
                })
                if cmd.startswith("diagnose:"):
                    diagnosis_rewards.append(reward)
                elif cmd.startswith("fix:"):
                    fix_rewards.append(reward)
                if result.done:
                    break
            except Exception as e:
                logger.warning(f"Step error: {e}")
                step_rewards.append(-0.1)
                break

    total_reward = sum(step_rewards) if step_rewards else -1.0
    return {
        "prompt_ids": prompt_ids,
        "completion_ids": completion_ids,
        "logprobs": logprobs,
        "total_reward": total_reward,
        "diagnosis_reward": diagnosis_rewards[-1] if diagnosis_rewards else 0.0,
        "fix_reward": fix_rewards[-1] if fix_rewards else 0.0,
    }

print("✅ Rollout function defined")

## 7. GRPO Training

Train the agent using Group Relative Policy Optimization. The environment provides real K8s cluster interactions and Claude-based reward signals.

In [ ]:
# ---- Tokenizer ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ---- Environment ----
env = KubeSreGymEnv(base_url=ENV_URL)

# ---- Dataset (each entry triggers one episode) ----
dataset = Dataset.from_dict({"prompt": ["Diagnose and fix this Kubernetes incident."] * NUM_EPISODES})

# ---- Output directory ----
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = Path(f"outputs/k8s-sre-grpo-{timestamp}")
output_dir.mkdir(parents=True, exist_ok=True)

# ---- Reward functions ----
def reward_total(completions, **kwargs):
    rewards = kwargs.get("total_reward") if kwargs else None
    return [float(r) for r in rewards] if rewards else [0.0 for _ in completions]

def reward_diagnosis(completions, **kwargs):
    rewards = kwargs.get("diagnosis_reward") if kwargs else None
    return [float(r) for r in rewards] if rewards else [0.0 for _ in completions]

def reward_fix(completions, **kwargs):
    rewards = kwargs.get("fix_reward") if kwargs else None
    return [float(r) for r in rewards] if rewards else [0.0 for _ in completions]

# ---- Reward CSV logger ----
reward_log_path = output_dir / "reward_log.csv"
episode_counter = [0]
all_rewards = []

with open(reward_log_path, "w", newline="") as f:
    csv.writer(f).writerow(["episode", "total_reward", "diagnosis_reward", "fix_reward", "timestamp"])

def log_episode(total_r, diag_r, fix_r):
    episode_counter[0] += 1
    all_rewards.append(total_r)
    with open(reward_log_path, "a", newline="") as f:
        csv.writer(f).writerow([episode_counter[0], total_r, diag_r, fix_r, datetime.now().isoformat()])
    n = len(all_rewards)
    last10 = all_rewards[-10:]
    logger.info(f"Episode {episode_counter[0]}: reward={total_r:.2f} | "
                f"mean={sum(all_rewards)/n:.2f}, mean(10)={sum(last10)/len(last10):.2f}")

# ---- Rollout function ----
def rollout_func(prompts, trainer):
    results = {k: [] for k in ["prompt_ids", "completion_ids", "logprobs",
                                "total_reward", "diagnosis_reward", "fix_reward"]}
    for _ in prompts:
        ep = rollout_once(trainer, env, tokenizer, SYSTEM_PROMPT, MAX_TURNS)
        for k in results:
            results[k].append(ep[k])
        log_episode(ep["total_reward"], ep["diagnosis_reward"], ep["fix_reward"])
    return results

print(f"✅ Training setup complete. Output: {output_dir}")

In [ ]:
# ---- GRPO Config ----
grpo_config = GRPOConfig(
    use_vllm=True,
    vllm_mode="colocate",
    output_dir=str(output_dir),
    num_train_epochs=1,
    learning_rate=5e-6,
    gradient_accumulation_steps=4,
    per_device_train_batch_size=1,
    generation_batch_size=NUM_GENERATIONS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=512,
    logging_steps=1,
    save_strategy="steps",
    save_steps=10,
    temperature=0.8,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    save_total_limit=3,
)

# ---- LoRA Config ----
peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# ---- Create Trainer ----
trainer = GRPOTrainer(
    model=MODEL_ID,
    processing_class=tokenizer,
    reward_funcs=[reward_total, reward_diagnosis, reward_fix],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
    peft_config=peft_config,
)

print("✅ GRPOTrainer initialized")

In [ ]:
# ---- Train! ----
print("🚀 Starting GRPO training...")
print(f"   Model: {MODEL_ID}")
print(f"   Episodes: {NUM_EPISODES}")
print(f"   Rollouts per episode: {NUM_GENERATIONS}")
print(f"   Environment: {ENV_URL}")
print()

try:
    trainer.train()
finally:
    env.close()

# Save model
trainer.save_model(str(output_dir))
print(f"\n✅ Model saved to {output_dir}")

## 8. Reward Curves

Visualize training progress — total reward, diagnosis accuracy, and fix success over episodes.

In [ ]:
import matplotlib.pyplot as plt

# Read reward log
episodes, totals, diags, fixes = [], [], [], []
with open(reward_log_path) as f:
    reader = csv.reader(f)
    next(reader)  # skip header
    for row in reader:
        episodes.append(int(row[0]))
        totals.append(float(row[1]))
        diags.append(float(row[2]))
        fixes.append(float(row[3]))

# Rolling average
window = min(10, len(episodes))
def rolling_avg(vals):
    return [sum(vals[max(0,i-window):i+1]) / min(i+1, window) for i in range(len(vals))]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Total reward
ax1.plot(episodes, totals, alpha=0.3, color="blue", label="Per episode")
ax1.plot(episodes, rolling_avg(totals), color="blue", linewidth=2, label=f"Rolling avg ({window})")
ax1.set_ylabel("Total Reward")
ax1.set_title("K8s SRE Agent — GRPO Training Rewards")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color="gray", linestyle="--", alpha=0.5)

# Diagnosis vs Fix
ax2.plot(episodes, rolling_avg(diags), color="orange", linewidth=2, label="Diagnosis (rolling)")
ax2.plot(episodes, rolling_avg(fixes), color="green", linewidth=2, label="Fix (rolling)")
ax2.set_xlabel("Episode")
ax2.set_ylabel("Reward")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / "reward_plot.png", dpi=150)
plt.show()
print(f"📊 Reward plot saved to {output_dir / 'reward_plot.png'}")

## 9. Push to Hub (Optional)

In [ ]:
# Uncomment to push trained model to HuggingFace Hub
# trainer.push_to_hub(repo_id=HUB_REPO)
# print(f"✅ Model pushed to https://huggingface.co/{HUB_REPO}")